In [1]:
# Clean, pinned install for Kaggle
%pip install -q --upgrade --upgrade-strategy only-if-needed \
  "gymnasium[atari,accept-rom-license]==1.2.0" \
  "stable-baselines3[extra]==2.7.0" \
  "ale-py==0.11.2" \
  "autorom==0.6.1" \
  "shimmy==2.0.0" \
  "imageio==2.37.0" \
  "pandas==2.2.2" \
  "matplotlib==3.10.0"

!AutoROM --accept-license

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.8/315.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 92.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adv

In [2]:
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
print("ALE/Pong-v5 available:", "ALE/Pong-v5" in gym.envs.registry)

ALE/Pong-v5 available: True


In [3]:
!mkdir -p /kaggle/working/Honorine-pong
!pwd
!ls -la /kaggle/working

/kaggle/working
total 68
drwxr-xr-x 3 root root  4096 Mar 19 09:36 .
drwxr-xr-x 8 root root  4096 Mar 19 09:35 ..
drwxr-xr-x 2 root root  4096 Mar 19 09:36 Honorine-pong
---------- 1 root root 54012 Mar 19 09:36 __notebook__.ipynb


In [4]:
%%writefile /kaggle/working/Honorine-pong/train.py
#!/usr/bin/env python3
"""
Train DQN on ALE/Pong-v5 with Stable-Baselines3.

This script supports:
- Comparing `CnnPolicy` vs `MlpPolicy`
- Logging reward/episode-length trends
- Saving one model as dqn_model.zip
- Appending experiment outcomes to a CSV table
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path
from typing import Dict, List

import ale_py
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecMonitor, VecTransposeImage


def register_ale_envs():
    """Required on some Kaggle/Gymnasium setups so ALE/* ids are visible."""
    gym.register_envs(ale_py)


def build_env_fn(env_id: str, monitor_path: Path, flatten_obs: bool, render_mode: str | None = None):
    """Create one Atari env with consistent preprocessing."""

    def _make():
        env = gym.make(env_id, render_mode=render_mode)
        try:
            env = AtariWrapper(
                env,
                noop_max=30,
                frame_skip=4,
                screen_size=84,
                terminal_on_life_loss=False,
                clip_reward=True,   # use False in play.py
                grayscale_obs=True,
                scale_obs=False,
            )
        except TypeError:
            # Older SB3 AtariWrapper signature (Kaggle often has this)
            env = AtariWrapper(
                env,
                noop_max=30,
                frame_skip=4,
                screen_size=84,
                terminal_on_life_loss=False,
                clip_reward=True,   # use False in play.py
            )

        if flatten_obs:
            env = gym.wrappers.FlattenObservation(env)
        env = Monitor(env, filename=str(monitor_path))
        return env

    return _make


def make_vec_env(env_id: str, policy_kind: str, monitor_path: Path, render_mode: str | None = None):
    flatten_obs = policy_kind.lower() == "mlp"
    vec_env = DummyVecEnv([build_env_fn(env_id, monitor_path, flatten_obs, render_mode=render_mode)])
    vec_env = VecMonitor(vec_env)
    if policy_kind.lower() == "cnn":
        # Stack 4 frames to provide temporal context for the CNN.
        vec_env = VecFrameStack(vec_env, n_stack=4)
        # SB3 CNN expects channel-first (C, H, W).
        vec_env = VecTransposeImage(vec_env)
    return vec_env


def load_monitor_dataframe(monitor_csv: Path) -> pd.DataFrame:
    if not monitor_csv.exists():
        return pd.DataFrame(columns=["r", "l", "t"])
    # First line in monitor.csv is metadata, second line is header.
    return pd.read_csv(monitor_csv, skiprows=1)


def save_training_plots(monitor_csv: Path, output_png: Path, policy_name: str):
    df = load_monitor_dataframe(monitor_csv)
    if df.empty:
        return

    rewards = df["r"].to_numpy()
    ep_len = df["l"].to_numpy()
    x = np.arange(1, len(df) + 1)

    window = min(25, len(df))
    reward_ma = pd.Series(rewards).rolling(window=window).mean()
    len_ma = pd.Series(ep_len).rolling(window=window).mean()

    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    fig.suptitle(f"{policy_name} Training Trends")

    axes[0].plot(x, rewards, alpha=0.35, label="episode reward")
    axes[0].plot(x, reward_ma, label=f"moving avg ({window})")
    axes[0].set_ylabel("Reward")
    axes[0].legend()

    axes[1].plot(x, ep_len, alpha=0.35, label="episode length")
    axes[1].plot(x, len_ma, label=f"moving avg ({window})")
    axes[1].set_ylabel("Episode Length")
    axes[1].set_xlabel("Episode")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(output_png, dpi=140)
    plt.close(fig)


def evaluate_model(model: DQN, env_id: str, policy_kind: str, n_episodes: int = 5) -> Dict[str, float]:
    eval_monitor = Path("tmp_eval_monitor.csv")
    eval_env = make_vec_env(env_id, policy_kind=policy_kind, monitor_path=eval_monitor)
    episode_rewards: List[float] = []

    for _ in range(n_episodes):
        obs = eval_env.reset()
        done = False
        total_reward = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)  # Greedy (max-Q) evaluation
            obs, rewards, dones, infos = eval_env.step(action)
            total_reward += float(rewards[0])
            done = bool(dones[0])
        episode_rewards.append(total_reward)

    eval_env.close()
    if eval_monitor.exists():
        eval_monitor.unlink()

    return {
        "eval_mean_reward": float(np.mean(episode_rewards)),
        "eval_std_reward": float(np.std(episode_rewards)),
    }


def append_experiment_row(results_csv: Path, row: Dict[str, str | int | float]):
    results_csv.parent.mkdir(parents=True, exist_ok=True)
    write_header = not results_csv.exists()
    with results_csv.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def train_one_policy(args: argparse.Namespace, policy_kind: str, run_dir: Path) -> Dict[str, float]:
    run_dir.mkdir(parents=True, exist_ok=True)
    monitor_csv = run_dir / "monitor.csv"
    eval_monitor_csv = run_dir / "eval_monitor.csv"

    env = make_vec_env(args.env_id, policy_kind=policy_kind, monitor_path=monitor_csv)
    eval_env = make_vec_env(args.env_id, policy_kind=policy_kind, monitor_path=eval_monitor_csv)

    policy_name = "CnnPolicy" if policy_kind.lower() == "cnn" else "MlpPolicy"

    model = DQN(
        policy_name,
        env,
        learning_rate=args.lr,
        gamma=args.gamma,
        batch_size=args.batch_size,
        buffer_size=args.buffer_size,
        learning_starts=args.learning_starts,
        train_freq=args.train_freq,
        gradient_steps=args.gradient_steps,
        target_update_interval=args.target_update_interval,
        exploration_initial_eps=args.epsilon_start,
        exploration_final_eps=args.epsilon_end,
        exploration_fraction=args.epsilon_decay_fraction,
        tensorboard_log=str(args.tensorboard_dir),
        verbose=1,
        seed=args.seed,
    )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(run_dir / "best_model"),
        log_path=str(run_dir / "eval_logs"),
        eval_freq=args.eval_freq,
        n_eval_episodes=args.eval_episodes,
        deterministic=True,
        render=False,
    )

    model.learn(total_timesteps=args.total_timesteps, callback=eval_callback, progress_bar=True)

    final_model_path = run_dir / "dqn_model.zip"
    model.save(str(final_model_path))
    save_training_plots(monitor_csv, run_dir / "training_curves.png", policy_name=policy_name)

    metrics = evaluate_model(model, args.env_id, policy_kind=policy_kind, n_episodes=args.eval_episodes)
    metrics["policy"] = policy_kind
    metrics["final_model_path"] = str(final_model_path)

    with (run_dir / "metrics.json").open("w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    env.close()
    eval_env.close()
    return metrics


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train DQN for Atari Pong (SB3 + Gymnasium).")
    parser.add_argument("--env-id", type=str, default="ALE/Pong-v5")
    parser.add_argument("--policy", type=str, default="both", choices=["cnn", "mlp", "both"])
    parser.add_argument("--total-timesteps", type=int, default=1_000_000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--exp-name", type=str, default="exp01")
    parser.add_argument("--output-dir", type=Path, default=Path("runs"))
    parser.add_argument("--tensorboard-dir", type=Path, default=Path("tb_logs"))
    parser.add_argument("--eval-freq", type=int, default=50_000)
    parser.add_argument("--eval-episodes", type=int, default=5)

    # Hyperparameters requested by assignment
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--gamma", type=float, default=0.99)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--epsilon-start", type=float, default=1.0)
    parser.add_argument("--epsilon-end", type=float, default=0.05)
    parser.add_argument(
        "--epsilon-decay-fraction",
        type=float,
        default=0.1,
        help="Fraction of total timesteps over which epsilon decays.",
    )

    # Useful defaults for Atari DQN
    parser.add_argument("--buffer-size", type=int, default=100_000)
    parser.add_argument("--learning-starts", type=int, default=20_000)
    parser.add_argument("--train-freq", type=int, default=4)
    parser.add_argument("--gradient-steps", type=int, default=1)
    parser.add_argument("--target-update-interval", type=int, default=10_000)

    return parser.parse_args()


def main():
    register_ale_envs()
    args = parse_args()
    base_dir = args.output_dir / args.exp_name
    base_dir.mkdir(parents=True, exist_ok=True)

    policies = ["cnn", "mlp"] if args.policy == "both" else [args.policy]
    summary_rows: List[Dict[str, str | int | float]] = []

    for policy_kind in policies:
        run_dir = base_dir / policy_kind
        metrics = train_one_policy(args, policy_kind=policy_kind, run_dir=run_dir)
        summary_rows.append(
            {
                "member_name": "Honorine",
                "experiment_name": args.exp_name,
                "policy": policy_kind,
                "env_id": args.env_id,
                "timesteps": args.total_timesteps,
                "lr": args.lr,
                "gamma": args.gamma,
                "batch_size": args.batch_size,
                "epsilon_start": args.epsilon_start,
                "epsilon_end": args.epsilon_end,
                "epsilon_decay_fraction": args.epsilon_decay_fraction,
                "eval_mean_reward": round(float(metrics["eval_mean_reward"]), 3),
                "eval_std_reward": round(float(metrics["eval_std_reward"]), 3),
                "noted_behavior": "Fill after run (e.g., stable learning, high variance, over/under-exploration).",
                "model_path": str(run_dir / "dqn_model.zip"),
            }
        )

    # Save one top-level model file for assignment convenience.
    # Prefer CNN model by default (usually better on visual Atari input).
    preferred_policy = "cnn" if "cnn" in policies else policies[0]
    preferred_model = base_dir / preferred_policy / "dqn_model.zip"
    assignment_model_path = Path("dqn_model.zip")
    if preferred_model.exists():
        assignment_model_path.write_bytes(preferred_model.read_bytes())

    results_csv = args.output_dir / "hyperparameter_results.csv"
    for row in summary_rows:
        append_experiment_row(results_csv, row)

    print("\nTraining complete.")
    print(f"Assignment model: {assignment_model_path.resolve()}")
    print(f"Results table updated: {results_csv.resolve()}")
    print("Per-policy outputs saved in:", base_dir.resolve())


if __name__ == "__main__":
    main()

Writing /kaggle/working/Honorine-pong/train.py


In [5]:
%%writefile /kaggle/working/Honorine-pong/play.py
#!/usr/bin/env python3
"""
Load a trained DQN model and run greedy evaluation episodes.

For local machines with GUI:
- use render mode "human" to watch gameplay in real time

For Kaggle/headless:
- use render mode "rgb_array" and optionally save a video
"""

from __future__ import annotations

import argparse
from pathlib import Path
from typing import List

import ale_py
import gymnasium as gym
import imageio.v2 as imageio
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage


def register_ale_envs():
    """Required on some Kaggle/Gymnasium setups so ALE/* ids are visible."""
    gym.register_envs(ale_py)


def build_env_fn(env_id: str, policy_kind: str, render_mode: str):
    def _make():
        env = gym.make(env_id, render_mode=render_mode)
        try:
            env = AtariWrapper(
                env,
                noop_max=30,
                frame_skip=4,
                screen_size=84,
                terminal_on_life_loss=False,
                clip_reward=True,   # use False in play.py
                grayscale_obs=True,
                scale_obs=False,
            )
        except TypeError:
            # Older SB3 AtariWrapper signature (Kaggle often has this)
            env = AtariWrapper(
                env,
                noop_max=30,
                frame_skip=4,
                screen_size=84,
                terminal_on_life_loss=False,
                clip_reward=True,   # use False in play.py
            )

        if policy_kind.lower() == "mlp":
            env = gym.wrappers.FlattenObservation(env)
        return env

    return _make


def make_vec_env(env_id: str, policy_kind: str, render_mode: str):
    vec_env = DummyVecEnv([build_env_fn(env_id, policy_kind, render_mode)])
    if policy_kind.lower() == "cnn":
        vec_env = VecFrameStack(vec_env, n_stack=4)
        vec_env = VecTransposeImage(vec_env)
    return vec_env


def main():
    register_ale_envs()
    parser = argparse.ArgumentParser(description="Play Atari Pong with a trained DQN model.")
    parser.add_argument("--env-id", type=str, default="ALE/Pong-v5")
    parser.add_argument("--model-path", type=Path, default=Path("dqn_model.zip"))
    parser.add_argument("--policy", type=str, choices=["cnn", "mlp"], default="cnn")
    parser.add_argument("--episodes", type=int, default=3)
    parser.add_argument("--render-mode", type=str, default="human", choices=["human", "rgb_array"])
    parser.add_argument("--save-video", action="store_true", help="Save video (useful for Kaggle).")
    parser.add_argument("--video-path", type=Path, default=Path("pong_agent.mp4"))
    args = parser.parse_args()

    if not args.model_path.exists():
        raise FileNotFoundError(f"Model not found: {args.model_path}")

    env = make_vec_env(args.env_id, args.policy, args.render_mode)
    model = DQN.load(str(args.model_path), env=env)

    all_returns: List[float] = []
    frames: List[np.ndarray] = []

    for episode in range(args.episodes):
        obs = env.reset()
        done = False
        ep_return = 0.0

        while not done:
            # deterministic=True = greedy action (max Q-value)
            action, _state = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = env.step(action)
            ep_return += float(rewards[0])
            done = bool(dones[0])

            if args.render_mode == "rgb_array" and args.save_video:
                frame = env.render()
                if isinstance(frame, np.ndarray):
                    frames.append(frame)

        all_returns.append(ep_return)
        print(f"Episode {episode + 1}/{args.episodes} return: {ep_return:.2f}")

    print("\nEvaluation done.")
    print(f"Mean return over {args.episodes} episodes: {np.mean(all_returns):.2f}")
    print(f"Std return: {np.std(all_returns):.2f}")

    if args.save_video and frames:
        imageio.mimsave(args.video_path, frames, fps=30)
        print(f"Saved video: {args.video_path.resolve()}")

    env.close()


if __name__ == "__main__":
    main()

Writing /kaggle/working/Honorine-pong/play.py


In [6]:
# Cell 5: Verify files exist
!ls -la /kaggle/working/Honorine-pong

total 28
drwxr-xr-x 2 root root  4096 Mar 19 09:36 .
drwxr-xr-x 3 root root  4096 Mar 19 09:36 ..
-rw-r--r-- 1 root root  4187 Mar 19 09:36 play.py
-rw-r--r-- 1 root root 11237 Mar 19 09:36 train.py


In [7]:
# Train
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp01 \
  --total-timesteps 1000000 \
  --lr 1e-4 \
  --gamma 0.99 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 09:36:26.437702: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773912986.616959      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773912986.669870      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773912987.094044      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773912987.094100      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773912987.094105      57 computation_placer.cc:177] computation placer alr

In [8]:
# Play + save video
!python /kaggle/working/Honorine-pong/play.py \
  --env-id ALE/Pong-v5 \
  --model-path /kaggle/working/dqn_model.zip \
  --policy cnn \
  --episodes 3 \
  --render-mode rgb_array \
  --save-video \
  --video-path /kaggle/working/pong_agent.mp4

2026-03-19 11:05:25.466334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773918325.488843      80 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773918325.496532      80 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773918325.524937      80 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773918325.524972      80 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773918325.524975      80 computation_placer.cc:177] computation placer alr

In [9]:
# SHOW VIDEO CELL
from IPython.display import Video
Video("/kaggle/working/pong_agent.mp4", embed=True)

In [10]:
# Cell 10: Download outputs from Kaggle working directory
# (run this near the end so files appear in right panel -> Output)
!ls -la /kaggle/working
!ls -la /kaggle/working/runs

total 47472
drwxr-xr-x 5 root root     4096 Mar 19 11:05 .
drwxr-xr-x 8 root root     4096 Mar 19 09:35 ..
-rw-r--r-- 1 root root 27259705 Mar 19 11:05 dqn_model.zip
drwxr-xr-x 2 root root     4096 Mar 19 09:36 Honorine-pong
---------- 1 root root 21048674 Mar 19 11:05 __notebook__.ipynb
-rw-r--r-- 1 root root   278202 Mar 19 11:05 pong_agent.mp4
drwxr-xr-x 3 root root     4096 Mar 19 11:05 runs
drwxr-xr-x 3 root root     4096 Mar 19 09:36 tb_logs
total 16
drwxr-xr-x 3 root root 4096 Mar 19 11:05 .
drwxr-xr-x 5 root root 4096 Mar 19 11:05 ..
drwxr-xr-x 3 root root 4096 Mar 19 09:36 exp01
-rw-r--r-- 1 root root  370 Mar 19 11:05 hyperparameter_results.csv


In [11]:
# Cell 11 (optional): zip important artifacts for submission
!zip -r /kaggle/working/honorine_pong_artifacts.zip /kaggle/working/Honorine-pong /kaggle/working/runs /kaggle/working/dqn_model.zip /kaggle/working/pong_agent.mp4

  adding: kaggle/working/Honorine-pong/ (stored 0%)
  adding: kaggle/working/Honorine-pong/play.py (deflated 63%)
  adding: kaggle/working/Honorine-pong/train.py (deflated 68%)
  adding: kaggle/working/runs/ (stored 0%)
  adding: kaggle/working/runs/hyperparameter_results.csv (deflated 30%)
  adding: kaggle/working/runs/exp01/ (stored 0%)
  adding: kaggle/working/runs/exp01/cnn/ (stored 0%)
  adding: kaggle/working/runs/exp01/cnn/best_model/ (stored 0%)
  adding: kaggle/working/runs/exp01/cnn/best_model/best_model.zip (stored 0%)
  adding: kaggle/working/runs/exp01/cnn/dqn_model.zip (stored 0%)
  adding: kaggle/working/runs/exp01/cnn/eval_monitor.csv (deflated 55%)
  adding: kaggle/working/runs/exp01/cnn/eval_logs/ (stored 0%)
  adding: kaggle/working/runs/exp01/cnn/eval_logs/evaluations.npz (deflated 59%)
  adding: kaggle/working/runs/exp01/cnn/monitor.csv (deflated 59%)
  adding: kaggle/working/runs/exp01/cnn/training_curves.png (deflated 5%)
  adding: kaggle/working/runs/exp01/cnn/m

In [12]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp01 \
  --total-timesteps 300000 \
  --lr 1e-4 \
  --gamma 0.99 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 11:06:10.801398: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773918370.823375     149 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773918370.829884     149 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773918370.846702     149 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773918370.846732     149 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773918370.846736     149 computation_placer.cc:177] computation placer alr

In [13]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp02 \
  --total-timesteps 300000 \
  --lr 5e-5 \
  --gamma 0.99 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 11:32:08.490372: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773919928.512122     172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773919928.518744     172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773919928.535864     172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773919928.535901     172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773919928.535907     172 computation_placer.cc:177] computation placer alr

In [14]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy mlp \
  --exp-name exp03 \
  --total-timesteps 300000 \
  --lr 1e-4 \
  --gamma 0.99 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 11:57:26.217366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773921446.239665     195 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773921446.246189     195 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773921446.263118     195 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773921446.263147     195 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773921446.263151     195 computation_placer.cc:177] computation placer alr

In [15]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp04 \
  --total-timesteps 300000 \
  --lr 2e-4 \
  --gamma 0.99 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 12:17:13.239132: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773922633.260930     218 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773922633.267680     218 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773922633.286115     218 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773922633.286150     218 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773922633.286153     218 computation_placer.cc:177] computation placer alr

In [16]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp05 \
  --total-timesteps 300000 \
  --lr 1e-4 \
  --gamma 0.95 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 12:43:04.969202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773924184.991318     241 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773924184.997991     241 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773924185.015895     241 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773924185.015932     241 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773924185.015936     241 computation_placer.cc:177] computation placer alr

In [17]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp06 \
  --total-timesteps 300000 \
  --lr 1e-4 \
  --gamma 0.999 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 13:08:41.427091: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773925721.449443     264 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773925721.455938     264 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773925721.473711     264 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773925721.473738     264 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773925721.473741     264 computation_placer.cc:177] computation placer alr

In [18]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp07 \
  --total-timesteps 300000 \
  --lr 1e-4 \
  --gamma 0.99 \
  --batch-size 64 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.10

2026-03-19 13:34:49.849775: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773927289.871858     287 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773927289.878465     287 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773927289.895704     287 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773927289.895732     287 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773927289.895736     287 computation_placer.cc:177] computation placer alr

In [19]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp08 \
  --total-timesteps 300000 \
  --lr 1e-4 \
  --gamma 0.99 \
  --batch-size 128 \
  --epsilon-start 1.0 \
  --epsilon-end 0.10 \
  --epsilon-decay-fraction 0.10

2026-03-19 14:01:33.060751: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773928893.084610     310 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773928893.091502     310 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773928893.109900     310 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773928893.109932     310 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773928893.109936     310 computation_placer.cc:177] computation placer alr

In [20]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp09 \
  --total-timesteps 1000000 \
  --lr 1e-4 \
  --gamma 0.99 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.01 \
  --epsilon-decay-fraction 0.20

2026-03-19 14:31:40.397133: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773930700.418270     333 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773930700.424849     333 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773930700.442540     333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773930700.442568     333 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773930700.442571     333 computation_placer.cc:177] computation placer alr

In [21]:
!python /kaggle/working/Honorine-pong/train.py \
  --env-id ALE/Pong-v5 \
  --policy cnn \
  --exp-name exp10 \
  --total-timesteps 1000000 \
  --lr 1e-4 \
  --gamma 0.99 \
  --batch-size 32 \
  --epsilon-start 1.0 \
  --epsilon-end 0.05 \
  --epsilon-decay-fraction 0.05

2026-03-19 15:59:27.106965: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773935967.129010     356 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773935967.135646     356 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773935967.155647     356 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773935967.155679     356 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773935967.155697     356 computation_placer.cc:177] computation placer alr